# Faults and ghost nodes in DISU

This notebook models a fault that dips at 45 degrees through four layers, with `vorflow.disu`:

1. **One mesh per layer.** Each layer is meshed with the fault at that layer's mid-depth (`add_line(..., is_barrier=True)`), so the fault is made of cell faces in every layer.
2. **Barrier connections.** `DisuGrid.barrier_faces` finds the connections the fault cuts and writes them as HFB input.
3. **Ghost nodes.** `estimate_gnc` corrects connections between vertically or laterally offset cells, without interpolating across the fault.
4. **A check** against the exact solution of a planar fault in a uniform aquifer.

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import LineString, box

from vorflow import ConceptualMesh, MeshGenerator, VoronoiTessellator, disu

# Import the meshing package before FloPy and its plotting backend.
import flopy
import matplotlib.pyplot as plt

mf6_exe = shutil.which(os.environ.get("MF6_EXE", "mf6"))
if mf6_exe is None:
    raise FileNotFoundError("MODFLOW 6 executable not found; set MF6_EXE or add mf6 to PATH.")
workspace = Path(tempfile.mkdtemp(prefix="vorflow_faults_"))

## 1. One mesh per layer

The domain is 200 m x 120 m with four flat layers 10 m thick. The fault strikes obliquely. Its trace at each layer's mid-depth is 10 m further east (measured across the strike) than the trace one layer up, so the fault plane dips at 45 degrees. Each layer is meshed with its own trace as a barrier line, so the vertical connections join different meshes.

In [ ]:
domain = box(0, 0, 200, 120)
normal = np.array([120.0, -40.0]) / np.hypot(120.0, 40.0)   # horizontal normal of the traces
shift = 10.0 / normal[0]                                     # x shift for 10 m across the strike


def trace(layer):
    return LineString([(70 + layer * shift, 0), (110 + layer * shift, 120)])


def mesh(line):
    cm = ConceptualMesh()
    cm.add_polygon(domain, zone_id=1, resolution=10)
    cm.add_line(line, line_id="fault", resolution=6, is_barrier=True)
    mg = MeshGenerator(background_lc=10, conceptual_mesh=cm, verbosity=0)
    mg.generate()
    return VoronoiTessellator(mg, cm).generate()


traces = [trace(k) for k in range(4)]
grids = [mesh(t) for t in traces]
d = disu.DisuGrid(grids, 40.0, [30.0, 20.0, 10.0, 0.0]).build()
print(d.report())

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, k in zip(axes, (0, 3)):
    grids[k].plot(ax=ax, facecolor="none", edgecolor="0.6", lw=0.4)
    for j, t in enumerate(traces):
        ax.plot(*t.xy, color="red" if j == k else "0.3", lw=2 if j == k else 1, ls="-" if j == k else "--")
    ax.set_title(f"layer {k}: its own trace (red), the others dashed")
    ax.set_aspect("equal")
plt.show()

## 2. Barrier connections

With one trace per layer, the fault in the model is a staircase:

- **walls**: horizontal connections that cross a layer's own trace. On a mesh built with `is_barrier=True` these faces lie on the trace (`on_trace`);
- **treads**: vertical connections whose two cells meet between the traces of two consecutive layers (inside the polygon joining them, closed at the fault tips). `faces.vertical` marks them.

`faces.factor` does not depend on K. It is |n<sub>fault</sub> · n<sub>face</sub>| for the plane through the traces, and the HFB exporters multiply the hydraulic characteristic by it. Then flow normal to the fault meets the plane's resistance on the staircase. Vertical barrier connections need MODFLOW 6 6.7 or later; the MODFLOW-USG export refuses them.

In [ ]:
faces = d.barrier_faces(traces)
print(f"{len(faces)} barrier connections: {(~faces.vertical).sum()} walls ({faces.on_trace.sum()} on a trace), "
      f"{faces.vertical.sum()} treads; factor {faces.factor.min():.3f} to {faces.factor.max():.3f}")

cells = d.cellmap[d.cellmap.node >= 0].sort_values("node")
layer, icpl = cells.layer.to_numpy(), cells.icpl.to_numpy()
plan = lambda node: grids[layer[node]].geometry.iloc[icpl[node]]  # noqa: E731
treads = [(i, j) for i, j, v in zip(faces.n, faces.m, faces.vertical) if v and {layer[i], layer[j]} == {0, 1}]
contacts = gpd.GeoSeries([plan(i).intersection(plan(j)) for i, j in treads])
fig, ax = plt.subplots(figsize=(7, 4))
grids[0].plot(ax=ax, facecolor="none", edgecolor="0.7", lw=0.4)
contacts.plot(ax=ax, facecolor="tab:blue", edgecolor="white", lw=0.3)
for k in (0, 1):
    ax.plot(*traces[k].xy, color="red" if k == 0 else "tab:orange", lw=2, label=f"layer {k} trace")
ax.set_title("Treads from layer 0 to 1: cut vertical contacts (blue)")
ax.set_aspect("equal")
ax.legend()
plt.show()

## 3. Ghost nodes

The layers and the meshes do not line up between layers, so many connections join offset cells. `estimate_gnc` gives each such connection a ghost head interpolated from immediate neighbours. With `barriers=faces.pairs` no ghost head is interpolated from across the fault: across a barrier the head jumps, so a neighbour there would bias the ghost. The barrier connections themselves are still corrected, each ghost from its own side. A connection whose only support lies across the fault is listed in `unsupported` and keeps the standard conductance.

In [ ]:
plain = d.estimate_gnc(connections="all")
aware = d.estimate_gnc(connections="all", barriers=faces.pairs)
for name, c in (("without barriers", plain), ("with barriers", aware)):
    print(f"{name}: {c.corrected_connections} connections corrected, {len(c.unsupported)} unsupported")

## 4. Check against the exact solution

In a uniform aquifer (K = 1 m/d) the flux q is the same on both sides of a planar fault, and the head drops by q·n / C across it, where C is the fault's hydraulic characteristic (here 0.01 /d). The flux here is normal to the fault, which the staircase represents exactly with the factor. The exact head is fixed on the boundary columns and on the top and bottom layers.

In [ ]:
xyz = np.column_stack((cells.x, cells.y, (cells.top + cells.bot) / 2))
a = np.array(traces[0].coords[0])
n3 = np.r_[normal, 1.0] / np.sqrt(2)                        # normal of the fault plane
far = (xyz[:, :2] - a) @ normal - (35.0 - xyz[:, 2]) > 0    # beyond the plane through the traces
C, q = 0.01, 0.01 * n3
exact = 50.0 - xyz @ q - far * (q @ n3) / C
x, y = xyz[:, 0], xyz[:, 1]
edge = (x < 15) | (x > 185) | (y < 15) | (y > 105) | (layer == 0) | (layer == 3)


def run(name, hydchr, gnc=None):
    sim = flopy.mf6.MFSimulation(sim_ws=str(workspace / name), exe_name=mf6_exe)
    flopy.mf6.ModflowTdis(sim)
    flopy.mf6.ModflowIms(sim, outer_dvclose=1e-10, inner_dvclose=1e-11, rcloserecord=1e-9,
                         linear_acceleration="bicgstab", inner_maximum=3000, outer_maximum=300)
    gwf = flopy.mf6.ModflowGwf(sim, modelname="m")
    flopy.mf6.ModflowGwfdisu(gwf, **d.get_gridprops_mf6_disu())
    flopy.mf6.ModflowGwfnpf(gwf, k=1.0)
    flopy.mf6.ModflowGwfic(gwf, strt=50.0)
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=[[(int(i),), float(exact[i])] for i in np.flatnonzero(edge)])
    flopy.mf6.ModflowGwfhfb(gwf, **hydchr)
    if gnc is not None:
        flopy.mf6.ModflowGwfgnc(gwf, **gnc.get_gridprops_mf6_gnc())
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord="m.hds", saverecord=[("HEAD", "ALL")])
    sim.write_simulation(silent=True)
    success, output = sim.run_simulation(silent=True, report=True)
    assert success, "\n".join(output[-20:])
    return np.abs(gwf.output.head().get_data().ravel() - exact)[~edge].max()


hfb = faces.get_gridprops_mf6_hfb(C)
no_factor = {**hfb, "stress_period_data": {0: [(i, j, C) for i, j, _ in hfb["stress_period_data"][0]]}}
print(f"head drop across the fault: {(q @ n3) / C:.2f} m")
for name, hydchr, gnc in (("HFB without the factor", no_factor, None), ("HFB", hfb, None),
                          ("HFB + GNC", hfb, plain), ("HFB + GNC with barriers", hfb, aware)):
    print(f"{name:26s} max head error {run(name.replace(' ', '_'), hydchr, gnc):.4f} m")

**Guidance:**

- **Keep the factor.** The exporters apply it; give the hydraulic characteristic (>= 0), not a conductance multiplier.
- **Pass `barriers=` to `estimate_gnc`** whenever the model has a barrier. Without it, ghost heads beside the fault average across the head jump.
- **GNC or XT3D, not both.** Next to offset cells, MODFLOW 6's XT3D treatment of HFB loses accuracy (seen with 6.8.2), so GNC with `barriers=` is the better choice for faults.
- **Flow along a dipping fault or unit.** Flat-topped cells make any dipping surface a staircase. Heads, flow direction and the flux per square metre can all be right while the total water crossing the unit is too low: the staircase removes about Δx·tanθ/b of the open area (column width Δx, dip θ, vertical thickness b). Keep Δx·tanθ/b at or below about 5 %, and compare total flows with a finer grid rather than trusting heads alone.
- **K-aware ghost nodes** (`method="k_aware"`) treat a face as a material contact only where K changes by at least `contrast` (default 10) across it. Recompute them whenever K changes.